# Passo 0 — Setup

Objetivo: preparar o ambiente, carregar `measured.dataset`, inspecionar o conjunto e visualizar exemplos da **máscara top-$k$** (heurística do Barino).

**Fonte de dados:** `paper/fbg-demodulated-lpfg/data/measured.dataset`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# classification/src no path (notebook em classification/notebooks/)
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data_utils import (
    FIGURES_DIR,
    K_DEFAULT,
    RANDOM_STATE,
    REPO_DATA,
    WL_RANGE,
    filter_wl_range,
    load_measured_dataset,
    make_topk_mask,
    normalize_input_strength,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_STATE)

print(f"RANDOM_STATE = {RANDOM_STATE}")
print(f"Dados: {REPO_DATA}")
print(f"Figuras: {FIGURES_DIR}")

## Versões do ambiente

In [ ]:
import matplotlib
import sklearn

try:
    import imblearn
    imblearn_v = imblearn.__version__
except ImportError:
    imblearn_v = "(não instalado — opcional no Passo 0)"

versions = pd.DataFrame(
    {
        "pacote": ["python", "numpy", "pandas", "scikit-learn", "matplotlib", "imbalanced-learn"],
        "versão": [
            sys.version.split()[0],
            np.__version__,
            pd.__version__,
            sklearn.__version__,
            matplotlib.__version__,
            imblearn_v,
        ],
    }
)
versions

## Carregar `measured.dataset`

In [ ]:
raw = load_measured_dataset()
X_raw = np.asarray(raw["input_strength"], dtype=float)
wl_bragg = np.asarray(raw["wl_bragg"], dtype=float)
y_res = np.asarray(raw["target"], dtype=float).ravel()

print(f"n (amostras)     = {X_raw.shape[0]}")
print(f"n_fbgs           = {X_raw.shape[1]}")
print(f"input_strength   : {X_raw.shape}, dtype={X_raw.dtype}")
print(f"wl_bragg         : {wl_bragg.shape}")
print(f"target (λ_res)   : {y_res.shape}")
print(f"λ_res min/max    : {y_res.min():.3f} / {y_res.max():.3f} nm")
print(f"NaNs em X        : {np.isnan(X_raw).sum()}")
print(f"NaNs em target   : {np.isnan(y_res).sum()}")

## Pré-visualização: normalização e filtro de faixa

No Passo 1 isso será formalizado e salvo. Aqui só inspecionamos o efeito.

In [ ]:
X_norm = normalize_input_strength(X_raw)
X_f, wl_f, y_f, keep = filter_wl_range(X_norm, wl_bragg, y_res, WL_RANGE)

print(f"Faixa mantida: ({WL_RANGE[0]}, {WL_RANGE[1]}) nm")
print(f"Amostras antes  : {len(y_res)}")
print(f"Amostras depois : {len(y_f)}  (removidas: {(~keep).sum()})")
print(f"Soma por linha (após norm) — min/max: {X_f.sum(axis=1).min():.6f} / {X_f.sum(axis=1).max():.6f}")

## Distribuição de $\lambda_{res}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(y_res, bins=40, color="#4c72b0", edgecolor="white")
axes[0].axvline(WL_RANGE[0], color="C3", ls="--", label="limites usados")
axes[0].axvline(WL_RANGE[1], color="C3", ls="--")
axes[0].set_xlabel(r"$\lambda_{res}$ (nm)")
axes[0].set_ylabel("Contagem")
axes[0].set_title("Todas as amostras")
axes[0].legend(fontsize=8)

axes[1].hist(y_f, bins=40, color="#55a868", edgecolor="white")
axes[1].set_xlabel(r"$\lambda_{res}$ (nm)")
axes[1].set_ylabel("Contagem")
axes[1].set_title(f"Após filtro {WL_RANGE[0]}–{WL_RANGE[1]} nm")

fig.tight_layout()
fig_path = FIGURES_DIR / "setup_lambda_res_hist.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Salvo: {fig_path}")

## Máscara top-$k$ (exemplo)

Rótulo multi-rótulo: $y_i=1$ nos $k$ FBGs com menor $|\lambda_{\mathrm{FBG},i}-\lambda_{res}|$.

In [ ]:
K = K_DEFAULT
Y_mask = make_topk_mask(wl_f, y_f, k=K)

print(f"k = {K}")
print(f"Y_mask shape     : {Y_mask.shape}")
print(f"1s por amostra   : min={Y_mask.sum(axis=1).min()}, max={Y_mask.sum(axis=1).max()}")
print(f"Frequência de 1 por FBG (fração das amostras):")
freq = Y_mask.mean(axis=0)
for i, f in enumerate(freq):
    print(f"  FBG[{i:2d}]: {f:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(np.arange(13), freq, color="#4c72b0", edgecolor="black", linewidth=0.4)
ax.axhline(K / 13, color="C3", ls="--", label=f"esperado se uniforme (k/13={K/13:.3f})")
ax.set_xticks(np.arange(13))
ax.set_xlabel("Índice do FBG")
ax.set_ylabel("Fração com rótulo 1")
ax.set_title(f"Balanceamento da máscara top-{K} (após filtro)")
ax.legend(fontsize=8)
fig.tight_layout()
fig_path = FIGURES_DIR / "setup_mask_frequency.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Salvo: {fig_path}")

In [ ]:
# Três exemplos: potências + posições FBG + máscara
rng = np.random.default_rng(RANDOM_STATE)
idxs = rng.choice(len(y_f), size=3, replace=False)

fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
for ax, i in zip(axes, idxs):
    ax.stem(wl_f[i], X_f[i], linefmt="C0-", markerfmt="C0o", basefmt="k-")
    active = Y_mask[i].astype(bool)
    ax.scatter(wl_f[i, active], X_f[i, active], s=80, facecolors="none", edgecolors="C3", linewidths=2, label="top-k (máscara)")
    ax.axvline(y_f[i], color="C2", ls="--", label=rf"$\lambda_{{res}}$={y_f[i]:.2f} nm")
    ax.set_ylabel("Potência norm.")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title(f"Amostra filtrada #{i}")

axes[-1].set_xlabel("Wavelength FBG (nm)")
fig.tight_layout()
fig_path = FIGURES_DIR / "setup_mask_examples.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Salvo: {fig_path}")

## Resumo do Passo 0

| Item | Valor |
|------|-------|
| Dataset | `measured.dataset` |
| $n$ bruto | ver saída acima |
| Dimensão de entrada | 13 (`input_strength`) |
| Rótulo | máscara multi-rótulo top-$k$ ($k=4$) |
| `RANDOM_STATE` | 42 |

**Próximo:** Passo 1 — pré-processamento formal, geração de rótulos e salvamento de artefatos em `results/`.